<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2018/---%20Day%2019%3A%20Go%20With%20The%20Flow%20---.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''#ip 0
seti 5 0 1
seti 6 0 2
addi 0 1 0
addr 1 2 3
setr 1 0 0
seti 8 0 4
seti 9 0 5'''
with open('19.txt') as f: puzzle = f.read()
puzzle

puzzle = puzzle.split('\n')
puzzle = [ line.split(' ') for line in puzzle ]
for line in puzzle: line[1:] = list(map(int, line[1:]))
puzzle

################################################################################
# PART 1

ip = puzzle[0][1]
instructions = puzzle[1:]

# set up program
registers = [0, 0, 0, 0, 0, 0]
# registers = [1, 0, 0, 0, 0, 0] # DONT EVEN TRY: program does not stop even after 40 mins
i = 0

# run program
iter = 0
while 0 <= i and i < len(instructions):

    # printing and failsafe for part 2
    iter += 1
    if iter < 100:
        print(registers)
    else:
        break

    # at the beginning of any step, set register bound to ip to ip's value
    registers[ip] = i

    # carry out instruction
    op = instructions[i][0]
    if op in ['addr', 'addi', 'mulr', 'muli', 'banr', 'bani', 'borr', 'bori']:
        a = registers[ instructions[i][1] ]
        b = instructions[i][2] if op[3] == 'i' else registers[ instructions[i][2] ]
        c = 0
        if op[:3] == 'add': c = a + b
        elif op[:3] == 'mul': c = a * b
        elif op[:3] == 'ban': c = a & b
        else: c = a | b
        registers[ instructions[i][3] ] = c
    if op in ['setr', 'seti']:
        a = instructions[i][1] if op[3] == 'i' else registers[ instructions[i][1] ]
        registers[ instructions[i][3] ] = a
    if op in ['gtir', 'gtri', 'gtrr', 'eqir', 'eqri', 'eqrr']:
        a = instructions[i][1] if op[2] == 'i' else registers[ instructions[i][1] ]
        b = instructions[i][2] if op[3] == 'i' else registers[ instructions[i][2] ]
        c = 0
        if op[:2] == 'gt': c = a > b
        else: c = a == b
        registers[ instructions[i][3] ] = int(c)

    i = registers[ip]
    i += 1

registers[0]

################################################################################
# PART 2

# 0     addi 1 16 1     R1 += 16
# 1     seti 1 5 3      R3 = 1
# 2     seti 1 7 5      R5 = 1
# ------------------------------------------------------------------------------
# 3     mulr 3 5 4      R4 = R3 * R5
# 4     eqrr 4 2 4      "If R2 == R3 * R5, then R0 += R3 # else do nothing"
# 5     addr 4 1 1      ...
# 6     addi 1 1 1	    ...
# 7     addr 3 0 0      ... endif
# 8     addi 5 1 5      R5 += 1
# 9     gtrr 5 2 4	    "If R5 > R4, then go to #12 else go back to #3"
# 10	addr 1 4 1	    ...
# 11	seti 2 1 1		... endif
# ------------------------------------------------------------------------------
# 12	addi 3 1 3	    R3 += 1
# 13	gtrr 3 2 4		"If R3 > R2, then go to #16 else go to #2"
# 14	addr 4 1 1		...
# 15	seti 1 3 1		... endif
# ------------------------------------------------------------------------------
# 16	mulr 1 1 1		R1 *= R1, abort program
# ------------------------------------------------------------------------------
# 17	addi 2 2 2      R2 += 2	"This part is entered only at the start of the program. Saves a number 1003 in R2"
# 18	mulr 2 2 2      R2 ^= 2
# 19	mulr 1 2 2      R2 *= R1
# 20	muli 2 11 2     R2 *= 11
# 21	addi 4 7 4      R4 += 7
# 22	mulr 4 1 4      R4 *= R1
# 23	addi 4 13 4     R4 += 13
# 24	addr 2 4 2      R2 += R4
# 25	addr 1 0 1		"If R0 == 1 then go to #27 # else go back to #1"
# 26	seti 0 9 1		... endif
# ------------------------------------------------------------------------------
# 27	setr 1 0 4      R4 = R1	"This part is entered only if R0 == 1 at the start of the program. Sets R2 to 10551403"
# 28	mulr 4 1 4      R4 *= R1
# 29	addr 1 4 4      R4 += R1
# 30	mulr 1 4 4      R4 *= R1
# 31	muli 4 14 4     R4 *= 14
# 32	mulr 4 1 4      R4 *= R1
# 33	addr 2 4 2      R2 += R4
# 34	seti 0 2 0      R0 = 0
# 35	seti 0 0 1      Back to #1
# ------------------------------------------------------------------------------
# Overall the program calculates the sum of divisors of the number written into R2